# YOLO26n Segmentation Model Experiment

This notebook trains, validates, and tests a YOLO26n segmentation model on the RoadVis/Pothole dataset.

The goal is to evaluate YOLO26n performance across different hyperparameter settings (learning rate, image size, weight decay), then run a final 100-epoch training on the best configuration.

## Setup Instructions

Place your dataset in:

```text
~/yolo_seg_project/
├── data.yaml
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

## Imports

In [ ]:
import time
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

from ultralytics import YOLO

## Device Setup

In [ ]:
device = 0 if torch.cuda.is_available() else "cpu"
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("GPU not available. Training will run on CPU.")

## Project and Dataset Paths

In [ ]:
project_root     = Path.home() / "Desktop" / "yolo_seg_project"
data_yaml        = project_root / "data.yaml"
train_images_dir = project_root / "images" / "train"
val_images_dir   = project_root / "images" / "val"
test_images_dir  = project_root / "images" / "test"
train_labels_dir = project_root / "labels" / "train"
val_labels_dir   = project_root / "labels" / "val"
test_labels_dir  = project_root / "labels" / "test"

results_dir = project_root / "results"
runs_dir    = project_root / "runs"
plots_dir   = results_dir  / "plots"

for d in [results_dir, runs_dir, plots_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Dataset YAML:", data_yaml)
print("Results folder:", results_dir)
print("Runs folder:",    runs_dir)

if not data_yaml.exists():
    raise FileNotFoundError(
        f"data.yaml not found at {data_yaml}.\n"
        "Please place your dataset inside ~/yolo_seg_project/"
    )

## Dataset Sanity Check

In [ ]:
def count_files(folder, exts):
    return sum(len(list(folder.glob(e))) for e in exts)

img_exts = ["*.jpg", "*.jpeg", "*.png"]
counts = {
    "train_images": count_files(train_images_dir, img_exts),
    "val_images":   count_files(val_images_dir,   img_exts),
    "test_images":  count_files(test_images_dir,  img_exts),
    "train_labels": count_files(train_labels_dir, ["*.txt"]),
    "val_labels":   count_files(val_labels_dir,   ["*.txt"]),
    "test_labels":  count_files(test_labels_dir,  ["*.txt"]),
}

for k, v in counts.items():
    print(f"{k}: {v}")

for split in ["train", "val", "test"]:
    if counts[f"{split}_images"] != counts[f"{split}_labels"]:
        print(f"WARNING: {split} image/label count mismatch!")
    else:
        print(f"{split}: OK — {counts[f'{split}_images']} image/label pairs")

## Image and Label Matching Check

In [ ]:
def inspect_sample_labels(images_dir, labels_dir, n=3):
    image_files = sorted(f for e in ["*.jpg","*.jpeg","*.png"] for f in images_dir.glob(e))
    if not image_files:
        print(f"No images found in: {images_dir}")
        return
    for img_path in image_files[:n]:
        label_path = labels_dir / f"{img_path.stem}.txt"
        print("Image:", img_path.name)
        print("Label:", label_path.name, "—", "EXISTS" if label_path.exists() else "MISSING")
        if label_path.exists():
            lines = label_path.read_text().splitlines()
            for line in lines[:2]:
                print(" ", line.strip())
        print("-" * 40)

inspect_sample_labels(train_images_dir, train_labels_dir, n=3)

## Helper Functions

In [ ]:
def extract_yolo_metrics(metrics, run_id, config, train_time_min):
    result = {
        "run_id":         run_id,
        "run_type":       config.get("run_type"),
        "model":          config["model"],
        "epochs":         config["epochs"],
        "imgsz":          config["imgsz"],
        "batch":          config["batch"],
        "optimizer":      config["optimizer"],
        "lr0":            config["lr0"],
        "weight_decay":   config["weight_decay"],
        "patience":       config["patience"],
        "augmentation":   config.get("augmentation"),
        "notes":          config.get("notes"),
        "train_time_min": train_time_min,
    }
    if hasattr(metrics, "box") and metrics.box is not None:
        result["box_precision"]  = float(metrics.box.mp)
        result["box_recall"]     = float(metrics.box.mr)
        result["box_map50"]      = float(metrics.box.map50)
        result["box_map50_95"]   = float(metrics.box.map)
    if hasattr(metrics, "seg") and metrics.seg is not None:
        result["seg_precision"]  = float(metrics.seg.mp)
        result["seg_recall"]     = float(metrics.seg.mr)
        result["seg_map50"]      = float(metrics.seg.map50)
        result["seg_map50_95"]   = float(metrics.seg.map)
    return result


def validate_yolo_run(config, train_time_min):
    best_weights = runs_dir / config["run_id"] / "weights" / "best.pt"
    if not best_weights.exists():
        raise FileNotFoundError(f"best.pt not found at: {best_weights}")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    model = YOLO(str(best_weights))
    metrics = model.val(data=str(data_yaml), device=device, batch=1)
    return extract_yolo_metrics(metrics, config["run_id"], config, train_time_min), metrics


def run_yolo_experiment(config):
    print(f"\n{'='*60}")
    print(f"Starting: {config['run_id']}  |  {config.get('notes','')}")
    print(f"{'='*60}")
    start = time.time()

    model = YOLO(config["model"])
    model.train(
        data         = str(data_yaml),
        epochs       = config["epochs"],
        imgsz        = config["imgsz"],
        batch        = config["batch"],
        device       = device,
        optimizer    = config["optimizer"],
        lr0          = config["lr0"],
        weight_decay = config["weight_decay"],
        patience     = config["patience"],
        project      = str(runs_dir),
        name         = config["run_id"],
        exist_ok     = True,
        amp          = True,
        workers      = 0,

    )

    elapsed = (time.time() - start) / 60.0
    print(f"Training finished in {elapsed:.1f} min")

    run_result, metrics = validate_yolo_run(config, elapsed)

    csv_path = results_dir / "experiment_log_yolo26.csv"
    df_new = pd.DataFrame([run_result])
    if csv_path.exists():
        df_existing = pd.read_csv(csv_path)
        df_new = pd.concat([df_existing, df_new], ignore_index=True)
    df_new = df_new.drop_duplicates(subset=["run_id"], keep="last")
    df_new.to_csv(csv_path, index=False)
    print(f"Saved log → {csv_path}")

    return run_result, metrics


def plot_training_history(run_id, title_prefix=None):
    title_prefix = title_prefix or run_id
    results_csv = runs_dir / run_id / "results.csv"
    if not results_csv.exists():
        raise FileNotFoundError(f"results.csv not found: {results_csv}")
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]

    # Loss curves
    plt.figure(figsize=(8, 4))
    for col, label in [("train/box_loss","Train Box"),("train/seg_loss","Train Seg"),
                        ("val/box_loss","Val Box"),("val/seg_loss","Val Seg")]:
        if col in df.columns:
            plt.plot(df["epoch"], df[col], marker="o", label=label)
    plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.title(f"{title_prefix} — Loss Curves")
    plt.legend(); plt.grid(True)
    plt.savefig(plots_dir / f"{run_id}_loss.png", bbox_inches="tight"); plt.show()

    # mAP curves
    plt.figure(figsize=(8, 4))
    for col, label in [("metrics/mAP50(M)","Mask mAP50"),("metrics/mAP50-95(M)","Mask mAP50-95")]:
        if col in df.columns:
            plt.plot(df["epoch"], df[col], marker="o", label=label)
    plt.xlabel("Epoch"); plt.ylabel("Score")
    plt.title(f"{title_prefix} — Mask mAP")
    plt.legend(); plt.grid(True)
    plt.savefig(plots_dir / f"{run_id}_map.png", bbox_inches="tight"); plt.show()

    # Precision / Recall
    plt.figure(figsize=(8, 4))
    for col, label in [("metrics/precision(M)","Mask Precision"),("metrics/recall(M)","Mask Recall")]:
        if col in df.columns:
            plt.plot(df["epoch"], df[col], marker="o", label=label)
    plt.xlabel("Epoch"); plt.ylabel("Score")
    plt.title(f"{title_prefix} — Mask Precision & Recall")
    plt.legend(); plt.grid(True)
    plt.savefig(plots_dir / f"{run_id}_pr.png", bbox_inches="tight"); plt.show()

## Experiment E00 — Sanity Check

3-epoch smoke test to verify the pipeline, GPU, and dataset are working correctly before committing to longer runs.

In [ ]:
config_e00 = {
    "run_id":       "E00_sanity",
    "run_type":     "sanity",
    "model":        "yolo26n-seg.pt",
    "epochs":       3,
    "imgsz":        640,
    "batch":        8,
    "patience":     3,
    "optimizer":    "AdamW",
    "lr0":          1e-3,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "Sanity check — 3 epochs to verify GPU and pipeline",
}

result_e00, metrics_e00 = run_yolo_experiment(config_e00)
plot_training_history("E00_sanity", "E00 Sanity")
result_e00

## Experiment E01 — Baseline

Standard 10-epoch baseline using YOLO26n pretrained weights with default hyperparameters. This is the reference point for all tuning experiments.

In [ ]:
config_e01 = {
    "run_id":       "E01_baseline",
    "run_type":     "baseline",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        640,
    "batch":        8,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          1e-3,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "Baseline — pretrained yolo26n, default lr0=1e-3",
}

result_e01, metrics_e01 = run_yolo_experiment(config_e01)
plot_training_history("E01_baseline", "E01 Baseline")
result_e01

## Experiment E02 — Learning Rate 1e-4

Lower learning rate to test if slower convergence improves generalization.

In [ ]:
config_e02 = {
    "run_id":       "E02_lr1e4",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        640,
    "batch":        8,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          1e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "LR tuning — lr0=1e-4",
}

result_e02, metrics_e02 = run_yolo_experiment(config_e02)
plot_training_history("E02_lr1e4", "E02 lr0=1e-4")
result_e02

## Experiment E03 — Learning Rate 3e-4

Mid-range learning rate — a common sweet spot for fine-tuning pretrained YOLO models.

In [ ]:
config_e03 = {
    "run_id":       "E03_lr3e4",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        640,
    "batch":        8,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "LR tuning — lr0=3e-4",
}

result_e03, metrics_e03 = run_yolo_experiment(config_e03)
plot_training_history("E03_lr3e4", "E03 lr0=3e-4")
result_e03

## Experiment E04 — Image Size 512

Smaller image size for faster training. Uses best lr0 from E02/E03.

In [ ]:
config_e04 = {
    "run_id":       "E04_imgsz512",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        512,
    "batch":        8,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "Image size tuning — imgsz=512, lr0=3e-4",
}

result_e04, metrics_e04 = run_yolo_experiment(config_e04)
plot_training_history("E04_imgsz512", "E04 imgsz=512")
result_e04

## Experiment E05 — Image Size 800

Larger image size for finer detail in pothole masks. More VRAM required.

In [ ]:
config_e05 = {
    "run_id":       "E05_imgsz800",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        800,
    "batch":        4,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "Image size tuning — imgsz=800, batch reduced to 4 for VRAM",
}

result_e05, metrics_e05 = run_yolo_experiment(config_e05)
plot_training_history("E05_imgsz800", "E05 imgsz=800")
result_e05

## Experiment E06 — Weight Decay 5e-4

Stronger regularization to reduce overfitting.

In [ ]:
config_e06 = {
    "run_id":       "E06_wd5e4",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        640,
    "batch":        8,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 5e-4,
    "augmentation": "default",
    "notes":        "Weight decay tuning — weight_decay=5e-4",
}

result_e06, metrics_e06 = run_yolo_experiment(config_e06)
plot_training_history("E06_wd5e4", "E06 wd=5e-4")
result_e06

## Experiment E07 — Weight Decay 1e-3

Even stronger regularization.

In [ ]:
config_e07 = {
    "run_id":       "E07_wd1e3",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        640,
    "batch":        8,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 1e-3,
    "augmentation": "default",
    "notes":        "Weight decay tuning — weight_decay=1e-3",
}

result_e07, metrics_e07 = run_yolo_experiment(config_e07)
plot_training_history("E07_wd1e3", "E07 wd=1e-3")
result_e07

## Compare All Tuning Experiments

Review val segmentation mAP50-95 across all runs to select the best config for the final 100-epoch training.

In [ ]:
log_path = results_dir / "experiment_log_yolo26.csv"
df_all = pd.read_csv(log_path)
df_all = df_all.sort_values("run_id").reset_index(drop=True)

display(df_all[["run_id","run_type","imgsz","batch","lr0","weight_decay",
                "seg_precision","seg_recall","seg_map50","seg_map50_95","train_time_min"]])

# Plot seg_map50 across all runs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(df_all["run_id"], df_all["seg_map50"])
axes[0].set_title("Seg mAP50 by Run")
axes[0].set_xlabel("Run ID")
axes[0].set_ylabel("mAP50")
axes[0].tick_params(axis="x", rotation=45)
axes[0].grid(axis="y")

axes[1].bar(df_all["run_id"], df_all["seg_map50_95"])
axes[1].set_title("Seg mAP50-95 by Run")
axes[1].set_xlabel("Run ID")
axes[1].set_ylabel("mAP50-95")
axes[1].tick_params(axis="x", rotation=45)
axes[1].grid(axis="y")

plt.tight_layout()
plt.savefig(plots_dir / "all_runs_comparison.png", bbox_inches="tight")
plt.show()

# Print best run
best_row = df_all.dropna(subset=["seg_map50_95"]).sort_values("seg_map50_95", ascending=False).iloc[0]
print(f"\nBest run by seg_map50_95: {best_row['run_id']} ({best_row['seg_map50_95']:.4f})")

## Helper: Save to Log

Centralized function to append any run result to the experiment CSV.

In [ ]:
def save_to_log(run_result):
    csv_path = results_dir / "experiment_log_yolo26.csv"
    df_new = pd.DataFrame([run_result])
    if csv_path.exists():
        df_existing = pd.read_csv(csv_path)
        df_new = pd.concat([df_existing, df_new], ignore_index=True)
    df_new = df_new.drop_duplicates(subset=["run_id"], keep="last")
    df_new.to_csv(csv_path, index=False)
    print(f"Saved → {csv_path}")

## Experiment E08 — Official YOLO26n Loss Weights

Uses the official YOLO26n pretrained recipe loss weights: `box=5.63`, `cls=0.56`, `dfl=9.04`.
Combined with best settings so far: `imgsz=800`, `lr0=3e-4`.

In [ ]:
config_e08 = {
    "run_id":       "E08_official_loss",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        800,
    "batch":        4,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "Official YOLO26n loss weights: box=5.63, cls=0.56, dfl=9.04",
}

model_e08 = YOLO(config_e08["model"])
start = time.time()
model_e08.train(
    data=str(data_yaml), epochs=config_e08["epochs"], imgsz=config_e08["imgsz"],
    batch=config_e08["batch"], device=device, optimizer=config_e08["optimizer"],
    lr0=config_e08["lr0"], weight_decay=config_e08["weight_decay"],
    patience=config_e08["patience"], project=str(runs_dir), name=config_e08["run_id"],
    exist_ok=True, amp=True, workers=0,
    box=5.63, cls=0.56, dfl=9.04,
)
elapsed = (time.time() - start) / 60.0
result_e08, metrics_e08 = validate_yolo_run(config_e08, elapsed)
save_to_log(result_e08)
plot_training_history("E08_official_loss", "E08 Official Loss Weights")
result_e08

## Experiment E09 — Official Momentum & Weight Decay

Uses official YOLO26n optimizer settings: `momentum=0.947`, `weight_decay=6.4e-4`.

In [ ]:
config_e09 = {
    "run_id":       "E09_official_optimizer",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        800,
    "batch":        4,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 6.4e-4,
    "augmentation": "default",
    "notes":        "Official YOLO26n optimizer: momentum=0.947, weight_decay=6.4e-4",
}

model_e09 = YOLO(config_e09["model"])
start = time.time()
model_e09.train(
    data=str(data_yaml), epochs=config_e09["epochs"], imgsz=config_e09["imgsz"],
    batch=config_e09["batch"], device=device, optimizer=config_e09["optimizer"],
    lr0=config_e09["lr0"], weight_decay=config_e09["weight_decay"],
    patience=config_e09["patience"], project=str(runs_dir), name=config_e09["run_id"],
    exist_ok=True, amp=True, workers=0,
    momentum=0.947,
)
elapsed = (time.time() - start) / 60.0
result_e09, metrics_e09 = validate_yolo_run(config_e09, elapsed)
save_to_log(result_e09)
plot_training_history("E09_official_optimizer", "E09 Official Momentum & WD")
result_e09

## Experiment E10 — Official Augmentation (copy_paste + scale)

Official YOLO26n recipe uses `copy_paste=0.075` and `scale=0.562`. Copy-paste augmentation pastes object instances onto new backgrounds, helping the model generalize to unseen pothole contexts.

In [ ]:
config_e10 = {
    "run_id":       "E10_official_augment",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        800,
    "batch":        4,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 1e-4,
    "augmentation": "copy_paste+scale",
    "notes":        "Official augmentation: copy_paste=0.075, scale=0.562",
}

model_e10 = YOLO(config_e10["model"])
start = time.time()
model_e10.train(
    data=str(data_yaml), epochs=config_e10["epochs"], imgsz=config_e10["imgsz"],
    batch=config_e10["batch"], device=device, optimizer=config_e10["optimizer"],
    lr0=config_e10["lr0"], weight_decay=config_e10["weight_decay"],
    patience=config_e10["patience"], project=str(runs_dir), name=config_e10["run_id"],
    exist_ok=True, amp=True, workers=0,
    copy_paste=0.075, scale=0.562,
)
elapsed = (time.time() - start) / 60.0
result_e10, metrics_e10 = validate_yolo_run(config_e10, elapsed)
save_to_log(result_e10)
plot_training_history("E10_official_augment", "E10 Official Augmentation")
result_e10

## Experiment E11 — MuSGD Optimizer (optimizer=auto)

YOLO26's native optimizer. Setting `optimizer=auto` lets Ultralytics select MuSGD automatically,
which is what was used to train the official YOLO26n pretrained weights.
Uses official `lr0=0.0054` designed for MuSGD.

In [ ]:
config_e11 = {
    "run_id":       "E11_musgd",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        800,
    "batch":        4,
    "patience":     5,
    "optimizer":    "auto",
    "lr0":          0.0054,
    "weight_decay": 6.4e-4,
    "augmentation": "default",
    "notes":        "MuSGD via optimizer=auto, official lr0=0.0054, wd=6.4e-4",
}

model_e11 = YOLO(config_e11["model"])
start = time.time()
model_e11.train(
    data=str(data_yaml), epochs=config_e11["epochs"], imgsz=config_e11["imgsz"],
    batch=config_e11["batch"], device=device, optimizer=config_e11["optimizer"],
    lr0=config_e11["lr0"], weight_decay=config_e11["weight_decay"],
    patience=config_e11["patience"], project=str(runs_dir), name=config_e11["run_id"],
    exist_ok=True, amp=True, workers=0,
    momentum=0.947, lrf=0.0495,
)
elapsed = (time.time() - start) / 60.0
result_e11, metrics_e11 = validate_yolo_run(config_e11, elapsed)
save_to_log(result_e11)
plot_training_history("E11_musgd", "E11 MuSGD Optimizer")
result_e11

## Experiment E12 — Cosine LR Decay

Cosine LR gradually reduces the learning rate following a cosine curve instead of linear decay.
Combined with best settings from E05/E08.

In [ ]:
config_e12 = {
    "run_id":       "E12_cos_lr",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        800,
    "batch":        4,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "Cosine LR decay: cos_lr=True, lrf=0.001",
}

model_e12 = YOLO(config_e12["model"])
start = time.time()
model_e12.train(
    data=str(data_yaml), epochs=config_e12["epochs"], imgsz=config_e12["imgsz"],
    batch=config_e12["batch"], device=device, optimizer=config_e12["optimizer"],
    lr0=config_e12["lr0"], weight_decay=config_e12["weight_decay"],
    patience=config_e12["patience"], project=str(runs_dir), name=config_e12["run_id"],
    exist_ok=True, amp=True, workers=0,
    cos_lr=True, lrf=0.001,
)
elapsed = (time.time() - start) / 60.0
result_e12, metrics_e12 = validate_yolo_run(config_e12, elapsed)
save_to_log(result_e12)
plot_training_history("E12_cos_lr", "E12 Cosine LR")
result_e12

## Compare All Experiments (E00 – E12)

Reads the full experiment log and plots all runs side by side.

In [ ]:
log_path = results_dir / "experiment_log_yolo26.csv"
df_all = pd.read_csv(log_path)
df_all["seg_map50_95"] = pd.to_numeric(df_all["seg_map50_95"], errors="coerce")
df_all["seg_map50"]    = pd.to_numeric(df_all["seg_map50"],    errors="coerce")
df_plot = df_all.dropna(subset=["seg_map50_95"]).sort_values("run_id").reset_index(drop=True)

# Summary table
cols = ["run_id","run_type","imgsz","batch","lr0","weight_decay",
        "seg_precision","seg_recall","seg_map50","seg_map50_95","notes"]
cols = [c for c in cols if c in df_plot.columns]
display(df_plot[cols])

# Bar chart — seg_map50 and seg_map50_95 side by side
x     = range(len(df_plot))
width = 0.35
fig, ax = plt.subplots(figsize=(16, 6))
bars1 = ax.bar([i - width/2 for i in x], df_plot["seg_map50"],    width, label="Seg mAP50",    color="steelblue")
bars2 = ax.bar([i + width/2 for i in x], df_plot["seg_map50_95"], width, label="Seg mAP50-95", color="coral")
ax.bar_label(bars1, fmt="%.3f", padding=2, fontsize=7)
ax.bar_label(bars2, fmt="%.3f", padding=2, fontsize=7)
ax.set_xticks(list(x))
ax.set_xticklabels(df_plot["run_id"], rotation=45, ha="right")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.0)
ax.set_title("YOLO26n All Experiments — Seg mAP50 vs mAP50-95")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(plots_dir / "all_runs_E00_E12_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

# Precision & Recall chart
fig2, ax2 = plt.subplots(figsize=(16, 5))
ax2.plot(df_plot["run_id"], pd.to_numeric(df_plot["seg_precision"], errors="coerce"),
         marker="o", label="Seg Precision", color="green")
ax2.plot(df_plot["run_id"], pd.to_numeric(df_plot["seg_recall"],    errors="coerce"),
         marker="s", label="Seg Recall",    color="orange")
ax2.set_xticks(range(len(df_plot)))
ax2.set_xticklabels(df_plot["run_id"], rotation=45, ha="right")
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1.0)
ax2.set_title("YOLO26n All Experiments — Seg Precision & Recall")
ax2.legend()
ax2.grid(linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(plots_dir / "all_runs_E00_E12_precision_recall.png", dpi=150, bbox_inches="tight")
plt.show()

# Best run
best = df_plot.sort_values("seg_map50_95", ascending=False).iloc[0]
print(f"\nBest run by seg_map50_95: {best['run_id']} ({best['seg_map50_95']:.4f})")
print(f"  lr0={best['lr0']}  imgsz={best['imgsz']}  weight_decay={best['weight_decay']}")

In [ ]:
config_e13 = {
    "run_id":       "E13_phase1_best",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        800,
    "batch":        4,
    "patience":     5,
    "optimizer":    "AdamW",
    "lr0":          3e-4,
    "weight_decay": 1e-4,
    "augmentation": "default",
    "notes":        "Phase 1 best combo: lr0=3e-4, imgsz=800, wd=1e-4",
}
result_e13, metrics_e13 = run_yolo_experiment(config_e13)
plot_training_history("E13_phase1_best", "E13 Phase 1 Best Combo")
result_e13

In [ ]:
config_e14 = {
    "run_id":       "E14_phase2_best",
    "run_type":     "tuning",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        800,
    "batch":        4,
    "patience":     5,
    "optimizer":    "auto",
    "lr0":          0.0054,
    "weight_decay": 6.4e-4,
    "augmentation": "musgd+coslr+augment",
    "notes":        "Phase 2 best combo: MuSGD, cos_lr, copy_paste, scale",
}
model_e14 = YOLO(config_e14["model"])
start = time.time()
model_e14.train(
    data         = str(data_yaml),
    epochs       = config_e14["epochs"],
    imgsz        = config_e14["imgsz"],
    batch        = config_e14["batch"],
    device       = device,
    optimizer    = config_e14["optimizer"],
    lr0          = config_e14["lr0"],
    weight_decay = config_e14["weight_decay"],
    patience     = config_e14["patience"],
    project      = str(runs_dir),
    name         = config_e14["run_id"],
    exist_ok     = True,
    amp          = True,
    workers      = 0,
    momentum     = 0.947,
    lrf          = 0.0495,
    cos_lr       = True,
    copy_paste   = 0.075,
    scale        = 0.562,
)
elapsed = (time.time() - start) / 60.0
result_e14, metrics_e14 = validate_yolo_run(config_e14, elapsed)
save_to_log(result_e14)
plot_training_history("E14_phase2_best", "E14 Phase 2 Best Combo")
result_e14

## Final Training

In [ ]:
log_path = results_dir / "experiment_log_yolo26.csv"
df_all   = pd.read_csv(log_path)

# Exclude sanity run from selection
df_tuning = df_all[df_all["run_type"] != "sanity"].copy()
df_tuning["seg_map50_95"] = pd.to_numeric(df_tuning["seg_map50_95"], errors="coerce")
best_row  = df_tuning.dropna(subset=["seg_map50_95"]).sort_values("seg_map50_95", ascending=False).iloc[0]

print(f"Best tuning run: {best_row['run_id']}")
print(f"  seg_map50_95 : {best_row['seg_map50_95']:.4f}")
print(f"  lr0          : {best_row['lr0']}")
print(f"  imgsz        : {int(best_row['imgsz'])}")
print(f"  weight_decay : {best_row['weight_decay']}")

config_final = {
    "run_id":       "E_FINAL_100ep",
    "run_type":     "final",
    "model":        "yolo26n-seg.pt",
    "epochs":       10,
    "imgsz":        int(best_row["imgsz"]),
    "batch":        int(best_row["batch"]),
    "patience":     5,
    "optimizer":    str(best_row["optimizer"]),
    "lr0":          float(best_row["lr0"]),
    "weight_decay": float(best_row["weight_decay"]),
    "augmentation": "default",
    "notes":        f"Final 100-epoch run using best config from {best_row['run_id']}",
}

print("\nFinal config:")
for k, v in config_final.items():
    print(f"  {k}: {v}")

In [ ]:
result_final, metrics_final = run_yolo_experiment(config_final)
plot_training_history("E_FINAL_100ep", "Final 100-Epoch Model")
result_final

## Final Test Set Evaluation

Run the final model on the unseen test split exactly once. This is the official reported score — never used during training or tuning.

In [ ]:
best_weights = runs_dir / "E_FINAL_100ep" / "weights" / "best.pt"
if not best_weights.exists():
    raise FileNotFoundError(f"best.pt not found: {best_weights}")

final_model  = YOLO(str(best_weights))
test_metrics = final_model.val(
    data   = str(data_yaml),
    split  = "test",
    imgsz  = config_final["imgsz"],
    device = device,
    batch  = 1,
)

test_result = {
    "run_id":               "E_FINAL_100ep",
    "best_weights":         str(best_weights),
    "test_box_precision":   float(test_metrics.box.mp),
    "test_box_recall":      float(test_metrics.box.mr),
    "test_box_map50":       float(test_metrics.box.map50),
    "test_box_map50_95":    float(test_metrics.box.map),
    "test_seg_precision":   float(test_metrics.seg.mp),
    "test_seg_recall":      float(test_metrics.seg.mr),
    "test_seg_map50":       float(test_metrics.seg.map50),
    "test_seg_map50_95":    float(test_metrics.seg.map),
}

df_test = pd.DataFrame([test_result])
test_csv = results_dir / "E_FINAL_unseen_test_results.csv"
df_test.to_csv(test_csv, index=False)
print(f"Saved test results → {test_csv}")
display(df_test)

## Final Test Metrics Bar Chart

In [ ]:
metric_cols = ["test_box_precision","test_box_recall","test_box_map50","test_box_map50_95",
               "test_seg_precision","test_seg_recall","test_seg_map50","test_seg_map50_95"]
vals  = [df_test.loc[0, c] for c in metric_cols]
names = [c.replace("test_","") for c in metric_cols]

plt.figure(figsize=(10, 5))
bars = plt.bar(names, vals)
plt.bar_label(bars, fmt="%.3f", padding=3)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Score")
plt.ylim(0, 1.1)
plt.title("Final Model — Unseen Test Metrics")
plt.grid(axis="y")
plt.savefig(plots_dir / "E_FINAL_test_metrics.png", bbox_inches="tight")
plt.show()

## Save Predictions on Test Images

In [ ]:
final_model.predict(
    source   = str(test_images_dir),
    imgsz    = config_final["imgsz"],
    conf     = 0.25,
    save     = True,
    project  = str(results_dir),
    name     = "E_FINAL_test_predictions",
    exist_ok = True,
)
print("Saved predictions to:", results_dir / "E_FINAL_test_predictions")